# Gestion des rallyes sportifs — alimentation de la base

Ce notebook :

1. recrée la base à partir des scripts SQL (`01` à `04`) ;
2. génère un jeu de données réaliste et volumineux (plusieurs dizaines d'éditions de rallyes, quelques milliers de concurrents, des dizaines de milliers de résultats) ;
3. l'insère en respectant toutes les contraintes d'intégrité, triggers compris ;
4. exécute les 7 requêtes du sujet (`05_requetes.sql`) ;
5. mesure l'effet des index avec `EXPLAIN ANALYZE`.

**Prérequis**

- la base PostgreSQL est démarrée (`docker compose up -d db`, voir le README) ;
- les bibliothèques Python sont installées : `pip install -r requirements.txt` ;
- ce notebook est dans le dossier `notebooks/` du projet, et les scripts SQL dans `sql/`.

Au premier `docker compose up`, ce notebook est exécuté automatiquement par le service `loader` ; une copie avec tous les résultats est enregistrée dans `notebooks/execution/`. On peut aussi l'ouvrir et l'exécuter à la main dans VS Code.

La génération utilise une graine fixe (`SEED`) : chaque exécution produit exactement les mêmes données.

In [ ]:
import os
import random
import re
import statistics
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import psycopg
from faker import Faker

# --- Connexion au conteneur Docker rallye-db ------------------------------
# Les variables d'environnement PG* permettent de changer de serveur
# sans modifier le notebook.
DB = dict(
    host=os.environ.get("PGHOST", "localhost"),
    port=int(os.environ.get("PGPORT", 5432)),
    dbname=os.environ.get("PGDATABASE", "rallyes"),
    user=os.environ.get("PGUSER", "rallye"),
    password=os.environ.get("PGPASSWORD", "rallye"),
)

# Dossier des scripts SQL : variable SQL_DIR (fixée par docker compose),
# sinon le dossier sql/ voisin de notebooks/
SQL_DIR = Path(os.environ.get("SQL_DIR", Path.cwd().parent / "sql"))
SEED = 42                 # graine : rend la génération reproductible

random.seed(SEED)
Faker.seed(SEED)
fake = Faker("fr_FR")
pd.set_option("display.max_rows", 40)


def connect(autocommit=False):
    return psycopg.connect(**DB, autocommit=autocommit)


def requete(sql, params=None):
    """Exécute une requête et renvoie le résultat sous forme de DataFrame."""
    with connect() as conn, conn.cursor() as cur:
        cur.execute(sql, params)
        if cur.description is None:
            return None
        df = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
        return df.convert_dtypes()   # entiers avec valeurs nulles affichés sans ".0"


print(requete("SELECT version()").iloc[0, 0])

## 1. Recréer la base

On exécute les scripts de création dans l'ordre. `01_tables.sql` supprime d'abord les tables existantes : le notebook peut donc être relancé autant de fois que nécessaire.

In [ ]:
SCRIPTS = ["01_tables.sql", "02_index.sql", "03_vues.sql", "04_fonctions_triggers.sql"]

with connect() as conn:
    for nom in SCRIPTS:
        conn.execute((SQL_DIR / nom).read_text(encoding="utf-8"))
        print("exécuté :", nom)

## 2. Données de référence

Pays (avec leurs villes), marques par catégorie de véhicule, sponsors, fournisseurs et itinéraires des rallyes.

Le **Paris-Dakar** garde ce nom pour toutes les éditions, de 1995 à 2026, même après son départ d'Afrique : les requêtes du sujet portent sur ce nom. Son itinéraire suit en revanche l'histoire réelle : Afrique jusqu'en 2008, Amérique du Sud de 2009 à 2019, Arabie saoudite depuis 2020.

In [ ]:
VILLES = {
    "France":          ["Paris", "Marseille", "Narbonne", "Perpignan", "Lyon"],
    "Espagne":         ["Barcelone", "Valence", "Grenade", "Almería"],
    "Maroc":           ["Tanger", "Er-Rachidia", "Ouarzazate", "Zagora", "Agadir", "Tan-Tan"],
    "Mauritanie":      ["Zouerate", "Atar", "Tidjikja", "Nouakchott", "Akjoujt"],
    "Mali":            ["Kayes", "Nioro", "Kiffa", "Tombouctou"],
    "Sénégal":         ["Dakar", "Saint-Louis", "Tambacounda", "Lac Rose", "Kaolack"],
    "Argentine":       ["Buenos Aires", "Córdoba", "San Juan", "Salta", "Fiambalá"],
    "Chili":           ["Copiapó", "Iquique", "Calama", "Antofagasta"],
    "Bolivie":         ["Uyuni", "Tupiza", "La Paz", "Oruro"],
    "Pérou":           ["Lima", "Pisco", "Arequipa", "San Juan de Marcona"],
    "Arabie saoudite": ["Djeddah", "Riyad", "Haïl", "Al-Ula", "Neom", "Shubaytah"],
    "Russie":          ["Moscou", "Astrakhan", "Volgograd"],
    "Kazakhstan":      ["Astana", "Aktobe", "Kyzylorda"],
    "Mongolie":        ["Oulan-Bator", "Dalanzadgad", "Bayankhongor"],
    "Chine":           ["Dunhuang", "Hami", "Urumqi"],
    "Italie":          ["Monaco", "Gênes"],
}

# Nationalités des concurrents : pays de VILLES + quelques autres, pondérés
NATIONALITES = {
    "France": 30, "Espagne": 10, "Italie": 6, "Argentine": 6, "Chili": 4,
    "Pays-Bas": 6, "Belgique": 5, "Allemagne": 5, "Portugal": 4,
    "Royaume-Uni": 4, "Russie": 4, "Afrique du Sud": 3, "Arabie saoudite": 2,
    "Sénégal": 2, "Maroc": 2, "États-Unis": 3, "Japon": 2, "Pologne": 2,
}
PAYS = sorted(set(VILLES) | set(NATIONALITES))

MARQUES_PAR_TYPE = {
    "Auto":   ["Toyota", "Mitsubishi", "Peugeot", "Citroën", "Mini", "Nissan",
               "Volkswagen", "Audi", "Ford", "BMW", "Honda", "Mazda"],
    "Moto":   ["KTM", "Honda", "Yamaha", "Husqvarna", "BMW", "Sherco", "Hero", "GasGas"],
    "Camion": ["Kamaz", "Iveco", "MAN", "Tatra", "Renault", "Hino", "Mercedes", "DAF"],
}
MARQUES = sorted({m for liste in MARQUES_PAR_TYPE.values() for m in liste})

SPONSORS = sorted({fake.unique.company()[:50] for _ in range(40)})
FOURNISSEURS = sorted({fake.unique.company()[:50] for _ in range(25)})


def itineraire(rallye, annee):
    """Pays traversés par une édition, dans l'ordre du parcours."""
    if rallye == "Paris-Dakar":
        if annee <= 2008:
            return ["France", "Espagne", "Maroc", "Mauritanie", "Mali", "Sénégal"]
        if annee <= 2019:
            return ["Argentine", "Chili", "Bolivie", "Pérou"]
        return ["Arabie saoudite"]
    if rallye == "Africa Eco Race":
        return ["Italie", "Maroc", "Mauritanie", "Sénégal"]
    if rallye == "Rallye du Maroc":
        return ["Maroc"]
    if rallye == "Silk Way Rally":
        return ["Russie", "Kazakhstan", "Mongolie", "Chine"]
    raise ValueError(rallye)


EDITIONS = (
    [("Paris-Dakar", a) for a in range(1995, 2027)]
    + [("Africa Eco Race", a) for a in range(2009, 2027)]
    + [("Rallye du Maroc", a) for a in range(2015, 2027)]
    + [("Silk Way Rally", a) for a in range(2016, 2022)]
)

print(len(PAYS), "pays,", len(MARQUES), "marques,", len(SPONSORS), "sponsors,",
      len(FOURNISSEURS), "fournisseurs,", len(EDITIONS), "éditions de rallyes")

## 3. Équipes

Chaque équipe engage une seule catégorie de véhicules (autos, camions ou motos).

- **Équipe officielle** : une par marque et par catégorie, elle ne représente que cette marque.
- **Association** : panachage de 2 ou 3 marques d'une même catégorie. Son nom est formé des marques, comme dans l'énoncé (`HONDA_MAZDA`).
- **Équipes privées** : petites structures engagées avec une seule marque, enregistrées comme équipes officielles. Elles donnent aux éditions une taille proche de celle du vrai Dakar (plusieurs centaines de véhicules).

La catégorie de chaque équipe n'est utilisée que pour la génération : en base, elle se déduit des véhicules.

In [ ]:
equipes = []            # (nomEquipe, budget, typeEquipe)
officielles = []        # (nomEquipe, nomMarque)
associations = []       # (nomEquipe,)
panache = []            # (nomEquipe, nomMarque)
marques_equipe = {}     # nomEquipe -> liste des marques autorisées (C14)
categorie_equipe = {}   # nomEquipe -> Auto / Moto / Camion

SUFFIXE = {"Auto": "Racing", "Moto": "Factory Team", "Camion": "Truck Team"}

for cat, marques in MARQUES_PAR_TYPE.items():
    # Équipes officielles : une par marque
    for m in marques:
        nom = "Toyota Gazoo Racing" if (m, cat) == ("Toyota", "Auto") else f"{m} {SUFFIXE[cat]}"
        budget = round(random.uniform(2e6, 15e6), -3)
        equipes.append((nom, budget, "OFFICIELLE"))
        officielles.append((nom, m))
        marques_equipe[nom] = [m]
        categorie_equipe[nom] = cat

    # Associations : panachages de 2 ou 3 marques de la même catégorie
    combinaisons = {("Honda", "Mazda")} if cat == "Auto" else set()   # exemple de l'énoncé
    while len(combinaisons) < 5:
        combinaisons.add(tuple(sorted(random.sample(marques, random.choice([2, 2, 3])))))
    for combo in sorted(combinaisons):
        nom = "_".join(m.upper() for m in combo)
        if nom in marques_equipe:          # même panachage déjà créé
            nom = f"{nom}_{cat.upper()}"
        budget = round(random.uniform(3e5, 3e6), -3)
        equipes.append((nom, budget, "ASSOCIATION"))
        associations.append((nom,))
        panache.extend((nom, m) for m in combo)
        marques_equipe[nom] = list(combo)
        categorie_equipe[nom] = cat

# Équipes privées : petites structures engagées avec une seule marque
SUFFIXE_PRIVE = ["Rallye Team", "Motorsport", "Raid Team", "Racing Team"]
for cat, nombre in {"Auto": 24, "Moto": 24, "Camion": 10}.items():
    for _ in range(nombre):
        nom = f"{fake.unique.last_name()} {random.choice(SUFFIXE_PRIVE)}"[:50]
        m = random.choice(MARQUES_PAR_TYPE[cat])
        equipes.append((nom, round(random.uniform(1e5, 1.5e6), -3), "OFFICIELLE"))
        officielles.append((nom, m))
        marques_equipe[nom] = [m]
        categorie_equipe[nom] = cat

sponsorise_equipe = [
    (nom, s) for nom, _, _ in equipes for s in random.sample(SPONSORS, random.randint(1, 4))
]

print(len(equipes), "équipes :", len(officielles), "officielles,", len(associations), "associations")

## 4. Concurrents

Un vivier de concurrents est tiré au sort à chaque édition. Comme il est plus petit que le nombre total de places, les mêmes concurrents reviennent d'une année à l'autre : c'est ce qui donne du sens à la requête 7.

In [ ]:
NB_CONCURRENTS = 8000

noms_nat, poids_nat = zip(*NATIONALITES.items())
concurrents = []        # (numConcurrent, nom, prenom, adresse, nomPays)
for num in range(1, NB_CONCURRENTS + 1):
    adresse = fake.address().replace("\n", ", ")[:100]
    concurrents.append((num, fake.last_name(), fake.first_name(), adresse,
                        random.choices(noms_nat, poids_nat)[0]))

print(len(concurrents), "concurrents")

## 5. Rallyes, étapes et résultats

Pour chaque édition :

- **Étapes** : 8 à 14 étapes enchaînées (la ville d'arrivée d'une étape est la ville de départ de la suivante). Une étape qui change de pays traverse les deux pays. Certaines spéciales sont des boucles (même ville au départ et à l'arrivée). La première étape est une liaison ; les suivantes sont des spéciales dans 70 % des cas.
- **Tranche horaire des spéciales** : départ à 7 h le jour de l'étape, durée maximale égale à 1,5 fois le temps prévu pour un camion (la catégorie la plus lente). Les heures sont stockées en temps epoch (secondes depuis le 1ᵉʳ janvier 1970).
- **Équipages** : chaque équipe participante engage 2 à 6 équipages de 2 concurrents tirés au sort. Toyota Gazoo Racing participe à tous les Paris-Dakar.
- **Véhicules** : chaque équipe a sa flotte, réutilisée d'une édition à l'autre (65 % de réutilisation), sans jamais affecter le même véhicule à deux équipages d'une même édition (C9).
- **Numéros des organisateurs** : motos à partir de 1, autos à partir de 200, camions à partir de 500 (C10).
- **Résultats** : la durée dépend de la distance, de la catégorie et d'un aléa (avec de rares incidents). Un équipage qui dépasse la tranche horaire d'une spéciale est disqualifié pour l'étape, sans durée enregistrée (C6, C7). Environ 1 % de disqualifications pour d'autres motifs, avec durée. Une partie des équipages abandonne en cours de rallye et n'a plus de résultat ensuite.

In [ ]:
VITESSE = {   # km/h moyens par catégorie : (spéciale, liaison)
    "Moto": (72, 95), "Auto": (82, 100), "Camion": (66, 85),
}
CYLINDREE = {
    "Moto":   lambda: random.choice([450, 450, 450, 500, 690]),
    "Auto":   lambda: random.randrange(2000, 6001, 100),
    "Camion": lambda: random.randrange(9000, 16501, 500),
}
PREMIER_NUMERO = {"Moto": 1, "Auto": 200, "Camion": 500}
PROBA_ABANDON = {"Moto": 0.30, "Auto": 0.20, "Camion": 0.18}

rallyes, etapes, speciales, liaisons, traverse = [], [], [], [], []
sponsorise_rallye, fournit = [], []
vehicules, equipages, compose, pilotes, resultats = [], [], [], [], []

flotte = defaultdict(list)       # nomEquipe -> numéros de série de ses véhicules
compteur_vehicule = 0


def nouveau_vehicule(equipe):
    global compteur_vehicule
    compteur_vehicule += 1
    cat = categorie_equipe[equipe]
    marque = random.choice(marques_equipe[equipe])
    serie = f"{marque[:3].upper()}-{cat[0]}-{compteur_vehicule:06d}"
    vehicules.append((serie, CYLINDREE[cat](), cat, marque))
    flotte[equipe].append(serie)
    return serie


def epoch(annee, jour, heure):
    """Instant (en secondes epoch) du jour `jour` de janvier de `annee`."""
    return int(datetime(annee, 1, jour, heure, tzinfo=timezone.utc).timestamp())


equipes_par_cat = defaultdict(list)
for nom, cat in categorie_equipe.items():
    equipes_par_cat[cat].append(nom)

for nomRallye, annee in EDITIONS:
    rallyes.append((nomRallye, annee))
    sponsorise_rallye += [(nomRallye, annee, s) for s in random.sample(SPONSORS, random.randint(2, 6))]
    fournit += [(nomRallye, annee, f) for f in random.sample(FOURNISSEURS, random.randint(2, 5))]

    # ---- Étapes ---------------------------------------------------------
    route = itineraire(nomRallye, annee)
    n_etapes = random.randint(10, 14) if nomRallye == "Paris-Dakar" else random.randint(8, 12)
    pays_etape = [route[min(len(route) - 1, i * len(route) // n_etapes)] for i in range(n_etapes + 1)]
    ville = random.choice(VILLES[pays_etape[0]])
    infos_etapes = []    # (numEtape, typeEtape, distance, borne_heures)
    for i in range(n_etapes):
        num = i + 1
        pays_dep, pays_arr = pays_etape[i], pays_etape[i + 1]
        type_etape = "LIAISON" if num == 1 or random.random() < 0.3 else "SPECIALE"
        derniere = num == n_etapes
        if derniere and "Sénégal" in route:
            arrivee = "Dakar"
        elif type_etape == "SPECIALE" and pays_dep == pays_arr and random.random() < 0.15:
            arrivee = ville                                   # boucle
        else:
            choix = [v for v in VILLES[pays_arr] if v != ville] or VILLES[pays_arr]
            arrivee = random.choice(choix)
        distance = round(random.uniform(150, 850) if type_etape == "SPECIALE"
                         else random.uniform(80, 500), 1)
        etapes.append((nomRallye, annee, num, ville, arrivee, distance, type_etape))
        for p in sorted({pays_dep, pays_arr}):
            traverse.append((p, nomRallye, annee, num))

        borne = None
        if type_etape == "SPECIALE":
            borne = 1.5 * distance / VITESSE["Camion"][0]     # heures autorisées
            debut = epoch(annee, num + 1, 7)
            speciales.append((nomRallye, annee, num, debut, debut + int(borne * 3600)))
        else:
            liaisons.append((nomRallye, annee, num))
        infos_etapes.append((num, type_etape, distance, borne))
        ville = arrivee

    # ---- Équipages ------------------------------------------------------
    participantes = (random.sample(equipes_par_cat["Auto"], random.randint(18, 26))
                     + random.sample(equipes_par_cat["Moto"], random.randint(18, 26))
                     + random.sample(equipes_par_cat["Camion"], random.randint(6, 10)))
    if nomRallye == "Paris-Dakar" and "Toyota Gazoo Racing" not in participantes:
        participantes.append("Toyota Gazoo Racing")

    engagements = [eq for eq in participantes for _ in range(random.randint(2, 6))]
    pilotes_edition = random.sample(range(1, NB_CONCURRENTS + 1), 2 * len(engagements))
    numero_orga = dict(PREMIER_NUMERO)
    deja_utilises = set()
    equipages_edition = []    # (numEquipage, catégorie, étape d'abandon ou None)

    for k, equipe in enumerate(engagements):
        num_eq = k + 1
        cat = categorie_equipe[equipe]
        equipages.append((nomRallye, annee, num_eq, equipe))
        compose.append((nomRallye, annee, pilotes_edition[2 * k], num_eq))
        compose.append((nomRallye, annee, pilotes_edition[2 * k + 1], num_eq))

        libres = [v for v in flotte[equipe] if v not in deja_utilises]
        serie = random.choice(libres) if libres and random.random() < 0.65 else nouveau_vehicule(equipe)
        deja_utilises.add(serie)
        pilotes.append((nomRallye, annee, num_eq, numero_orga[cat], serie))
        numero_orga[cat] += 1

        abandon = random.randint(2, n_etapes) if random.random() < PROBA_ABANDON[cat] else None
        equipages_edition.append((num_eq, cat, abandon))

    # ---- Résultats ------------------------------------------------------
    for num, type_etape, distance, borne in infos_etapes:
        for num_eq, cat, abandon in equipages_edition:
            if abandon is not None and num >= abandon:
                continue                                      # a abandonné
            vitesse = VITESSE[cat][0 if type_etape == "SPECIALE" else 1]
            alea = random.gauss(1.0, 0.10)
            if random.random() < 0.04:
                alea *= random.uniform(1.4, 2.2)              # incident mécanique
            duree = max(0.2, distance / vitesse * alea)
            if borne is not None and duree > borne:
                resultats.append((nomRallye, annee, num, num_eq, None, True))     # hors délai
            elif random.random() < 0.01:
                resultats.append((nomRallye, annee, num, num_eq, round(duree, 4), True))
            else:
                resultats.append((nomRallye, annee, num, num_eq, round(duree, 4), False))

volumes_generes = {
    "rallye": len(rallyes), "etape": len(etapes), "traverse": len(traverse),
    "vehicule": len(vehicules), "equipage": len(equipages),
    "compose": len(compose), "resultat": len(resultats),
}
volumes_generes

## 6. Insertion

Toutes les données sont insérées dans **une seule transaction**, dans l'ordre imposé par les clés étrangères. On utilise `COPY`, beaucoup plus rapide que des `INSERT` ligne par ligne.

Les triggers différés (C12 : 2 concurrents et un véhicule par équipage, C13 : au moins 2 marques par association, C15 : ligne de sous-type pour chaque étape et chaque équipe) sont vérifiés au `COMMIT`. Le trigger C14 (cohérence entre la marque du véhicule et celle de l'équipe) est vérifié à chaque insertion dans `pilote`. Si une seule contrainte est violée, toute la transaction est annulée et la base reste vide.

In [ ]:
ORDRE = [   # (table, colonnes, lignes) dans l'ordre des clés étrangères
    ("pays", "nomPays", [(p,) for p in PAYS]),
    ("marque", "nomMarque", [(m,) for m in MARQUES]),
    ("sponsor", "nomSponsor", [(s,) for s in SPONSORS]),
    ("fournisseur", "nomFournisseur", [(f,) for f in FOURNISSEURS]),
    ("equipe", "nomEquipe, budget, typeEquipe", equipes),
    ("officielle", "nomEquipe, nomMarque", officielles),
    ("association", "nomEquipe", associations),
    ("panache", "nomEquipe, nomMarque", panache),
    ("sponsorise_equipe", "nomEquipe, nomSponsor", sponsorise_equipe),
    ("concurrent", "numConcurrent, nom, prenom, adresse, nomPays", concurrents),
    ("rallye", "nomRallye, annee", rallyes),
    ("etape", "nomRallye, annee, numEtape, villeDepart, villeArrivee, distance, typeEtape", etapes),
    ("speciale", "nomRallye, annee, numEtape, heureDebutEpoch, heureFinEpoch", speciales),
    ("liaison", "nomRallye, annee, numEtape", liaisons),
    ("traverse", "nomPays, nomRallye, annee, numEtape", traverse),
    ("sponsorise_rallye", "nomRallye, annee, nomSponsor", sponsorise_rallye),
    ("fournit", "nomRallye, annee, nomFournisseur", fournit),
    ("vehicule", "numeroSerie, cylindree, type, nomMarque", vehicules),
    ("equipage", "nomRallye, annee, numEquipage, nomEquipe", equipages),
    ("compose", "nomRallye, annee, numConcurrent, numEquipage", compose),
    ("pilote", "nomRallye, annee, numEquipage, numeroOrga, numeroSerie", pilotes),
    ("resultat", "nomRallye, annee, numEtape, numEquipage, dureeHeure, disqualifiee", resultats),
]

with connect() as conn:            # une seule transaction, validée à la sortie du bloc
    with conn.cursor() as cur:
        for table, colonnes, lignes in ORDRE:
            with cur.copy(f"COPY {table} ({colonnes}) FROM STDIN") as copie:
                for ligne in lignes:
                    copie.write_row(ligne)
            print(f"{table:<18} {len(lignes):>7} lignes")
    print("COMMIT ... (vérification des triggers différés)")
print("Insertion terminée.")

# Met à jour les statistiques utilisées par l'optimiseur de requêtes
with connect(autocommit=True) as conn:
    conn.execute("VACUUM ANALYZE")

## 7. Vérification des volumes

In [ ]:
TABLES = [t for t, _, _ in ORDRE]
volumes = requete(" UNION ALL ".join(
    f"SELECT '{t}' AS table_, count(*) AS lignes FROM {t}" for t in TABLES))
volumes

In [ ]:
# Quelques indicateurs pour vérifier que les données font réagir les requêtes
requete("""
SELECT
  (SELECT count(*) FROM resultat WHERE disqualifiee)                        AS disqualifications,
  (SELECT count(*) FROM resultat WHERE disqualifiee AND dureeHeure IS NULL) AS hors_delai,
  (SELECT count(*) FROM etape et JOIN speciale USING (nomRallye, annee, numEtape)) AS speciales,
  (SELECT count(*) FROM etape WHERE villeDepart = villeArrivee)             AS boucles,
  (SELECT count(*) FROM (SELECT numeroSerie FROM pilote
                         GROUP BY numeroSerie HAVING count(*) > 1) x)       AS vehicules_reutilises,
  (SELECT round(avg(n), 2) FROM (SELECT count(*) n FROM compose
                                 GROUP BY numConcurrent) x)                 AS participations_moyennes
""")

## 8. Les 7 requêtes du sujet

Les requêtes sont lues directement dans `05_requetes.sql`, pour n'avoir qu'une seule version de chaque requête.

In [ ]:
def lire_requetes(chemin):
    """Découpe 05_requetes.sql en blocs commençant par '-- N)'."""
    texte = chemin.read_text(encoding="utf-8")
    blocs = re.split(r"(?m)^(?=-- \d\))", texte)[1:]
    requetes = []
    for bloc in blocs:
        lignes = bloc.strip().splitlines()
        titre = " ".join(l.lstrip("- ").strip() for l in lignes if l.startswith("--"))
        sql = "\n".join(l for l in lignes if not l.startswith("--")).strip().rstrip(";")
        requetes.append((titre, sql))
    return requetes


REQUETES = lire_requetes(SQL_DIR / "05_requetes.sql")
[t for t, _ in REQUETES]

In [ ]:
titre, sql = REQUETES[0]
print(titre)
df = requete(sql)
print(len(df), "ligne(s)")
df.head(20)

In [ ]:
titre, sql = REQUETES[1]
print(titre)
df = requete(sql)
print(len(df), "ligne(s)")
df.head(20)

In [ ]:
titre, sql = REQUETES[2]
print(titre)
df = requete(sql)
print(len(df), "ligne(s)")
df.head(20)

In [ ]:
titre, sql = REQUETES[3]
print(titre)
df = requete(sql)
print(len(df), "ligne(s)")
df.head(20)

In [ ]:
titre, sql = REQUETES[4]
print(titre)
df = requete(sql)
print(len(df), "ligne(s)")
df.head(20)

In [ ]:
titre, sql = REQUETES[5]
print(titre)
df = requete(sql)
print(len(df), "ligne(s)")
df.head(20)

In [ ]:
titre, sql = REQUETES[6]
print(titre)
df = requete(sql)
print(len(df), "ligne(s)")
df.head(20)

## 9. Effet des index

Pour chaque requête testée, on mesure le temps d'exécution réel (`EXPLAIN ANALYZE`) dans deux situations :

- **avec index** : la base telle qu'elle est créée par les scripts ;
- **sans index** : dans une transaction, on supprime les index de la table concernée (y compris la clé primaire, qui jouerait le même rôle), on mesure, puis on annule la transaction (`ROLLBACK`). La base n'est donc pas modifiée.

Chaque mesure est répétée plusieurs fois et on garde la médiane. La colonne « lecture de la table » indique comment PostgreSQL accède à la table principale : *Index Scan* ou *Bitmap* (via un index) ou *Seq Scan* (parcours complet).

In [ ]:
def plan(sql, suppressions=(), repetitions=7):
    """Renvoie (temps médian en ms, plan d'exécution) de la requête.
    `suppressions` : instructions exécutées avant la mesure, puis annulées."""
    temps, texte = [], ""
    with connect() as conn, conn.cursor() as cur:
        for instruction in suppressions:
            cur.execute(instruction)
        for _ in range(repetitions):
            cur.execute("EXPLAIN (ANALYZE, BUFFERS) " + sql)
            texte = "\n".join(r[0] for r in cur.fetchall())
            temps.append(float(re.search(r"Execution Time: ([\d.]+)", texte).group(1)))
        conn.rollback()          # remet les index supprimés
    return statistics.median(temps), texte


def acces(texte, table):
    """Mode de lecture de `table` dans un plan : Index Scan, Bitmap ou Seq Scan."""
    for ligne in texte.splitlines():
        if re.search(rf" on {table}\b", ligne):
            m = re.search(r"(Index Only Scan|Index Scan|Bitmap Heap Scan|Seq Scan)", ligne)
            if m:
                return m.group(1)
    return "?"


SANS_INDEX_RESULTAT = ["DROP INDEX idx_resultat_equipage",
                       "ALTER TABLE resultat DROP CONSTRAINT resultat_pkey"]

TESTS = [   # (description, table principale, requête, index supprimés)
    ("Résultats d'un équipage (écran de l'application)", "resultat",
     "SELECT * FROM resultat WHERE nomRallye = 'Paris-Dakar' AND annee = 2002 AND numEquipage = 12",
     SANS_INDEX_RESULTAT),
    ("Classement d'une étape (écran de l'application)", "resultat",
     "SELECT * FROM v_classement_etape WHERE nomRallye = 'Paris-Dakar' AND annee = 2002 "
     "AND numEtape = 5 ORDER BY disqualifiee, rang",
     SANS_INDEX_RESULTAT),
    ("Participations d'un concurrent (écran de l'application)", "compose",
     "SELECT * FROM v_participation WHERE numConcurrent = 1234",
     ["DROP INDEX idx_compose_concurrent"]),
    ("Requête 6 : classement complet du Paris-Dakar 2002", "resultat",
     REQUETES[5][1], SANS_INDEX_RESULTAT),
    ("Requête 3 : étapes au Sénégal", "traverse",
     REQUETES[2][1], ["ALTER TABLE traverse DROP CONSTRAINT traverse_pkey"]),
    ("Requête 5 : véhicules Toyota au Paris-Dakar", "vehicule",
     REQUETES[4][1], ["DROP INDEX idx_vehicule_marque", "DROP INDEX idx_pilote_vehicule"]),
]

mesures, plans = [], {}
for nom, table, sql, suppressions in TESTS:
    avec, plan_avec = plan(sql)
    sans, plan_sans = plan(sql, suppressions)
    plans[nom] = (plan_avec, plan_sans)
    mesures.append({
        "requête": nom,
        "avec index (ms)": round(avec, 3),
        "sans index (ms)": round(sans, 3),
        "gain (x)": round(sans / avec, 1),
        "lecture avec index": acces(plan_avec, table),
        "lecture sans index": acces(plan_sans, table),
    })

pd.DataFrame(mesures)

**Lecture des résultats**

- Les requêtes **sélectives**, qui ne lisent qu'une petite partie d'une grande table (les résultats d'un équipage, le classement d'une étape, les participations d'un concurrent), sont nettement accélérées : avec l'index, PostgreSQL va directement aux lignes utiles au lieu de parcourir toute la table.
- Pour la **requête 6**, la lecture de `resultat` est aussi beaucoup plus rapide avec l'index, mais la requête renvoie plus de 2 000 lignes : l'essentiel du temps est passé dans les jointures, le tri et le calcul du rang (`RANK()`), que l'index n'accélère pas.
- Les requêtes sur de **petites tables** (`traverse`, `vehicule` : quelques milliers de lignes au plus) ne gagnent rien, et l'accès par index peut même être légèrement plus lent : la table tient en quelques pages mémoire, et la parcourir entièrement coûte moins cher que de passer par l'index. Les écarts ne portent alors que sur quelques dixièmes de milliseconde. Ces index deviendraient utiles avec un volume beaucoup plus important.

Ci-dessous, le plan détaillé du classement d'une étape, avec et sans index (à reprendre dans le rapport).

In [ ]:
avec, sans = plans["Classement d'une étape (écran de l'application)"]
print("=== AVEC INDEX ===\n" + avec)
print("\n=== SANS INDEX ===\n" + sans)

### Utilisation des index

Statistiques de PostgreSQL : nombre de fois où chaque index créé par `02_index.sql` a été utilisé depuis le chargement. Les index des clés étrangères vers les petites tables de référence (sponsors, fournisseurs…) restent à 0 : ils servent surtout lors des suppressions dans ces tables, que ce notebook n'effectue pas.

In [ ]:
requete("""
SELECT indexrelname AS index, relname AS table_, idx_scan AS utilisations
FROM pg_stat_user_indexes
WHERE indexrelname LIKE 'idx\\_%'
ORDER BY idx_scan DESC
""")